In [8]:
import os
import pandas as pd
import snowflake.connector
from dotenv import load_dotenv

In [9]:
load_dotenv()

conn = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
    database=os.getenv('SNOWFLAKE_DATABASE'),
    schema=os.getenv('SNOWFLAKE_SCHEMA'),
    role=os.getenv('SNOWFLAKE_ROLE')
)

## 📦 Staging: `01_ORDERS`

**Source:** `SALES_RAW.01_ORDERS`

### Transformations

| Category | Description |
|----------|-------------|
| Timestamps | Parse all date columns to `TIMESTAMP` |
| Date Parts | Extract `year`, `quarter`, `month`, `week`, `day of week`, `hour` |
| Lifecycle Durations | Calculate `hours_to_approve`, `hours_to_ship`, `hours_in_transit` |
| Delivery Performance | Flag late deliveries, calculate `days_late` |
| Data Quality | Flag missing customers, impossible dates, missing delivery dates |

In [10]:
query = """DESCRIBE TABLE sales_raw."01_ORDERS"
"""
df_orders = pd.read_sql(query, conn)
df_orders.columns = df_orders.columns.str.lower()

print(df_orders)


                       name               type    kind null? default  \
0                  ORDER_ID  VARCHAR(16777216)  COLUMN     Y    None   
1               CUSTOMER_ID  VARCHAR(16777216)  COLUMN     Y    None   
2                ORDER_DATE  VARCHAR(16777216)  COLUMN     Y    None   
3             APPROVAL_DATE  VARCHAR(16777216)  COLUMN     Y    None   
4              SHIPPED_DATE  VARCHAR(16777216)  COLUMN     Y    None   
5            DELIVERED_DATE  VARCHAR(16777216)  COLUMN     Y    None   
6   ESTIMATED_DELIVERY_DATE  VARCHAR(16777216)  COLUMN     Y    None   
7              ORDER_STATUS  VARCHAR(16777216)  COLUMN     Y    None   
8           CUSTOMER_REGION  VARCHAR(16777216)  COLUMN     Y    None   
9             CUSTOMER_CITY  VARCHAR(16777216)  COLUMN     Y    None   
10                SELLER_ID  VARCHAR(16777216)  COLUMN     Y    None   
11                  CARRIER  VARCHAR(16777216)  COLUMN     Y    None   
12         WAREHOUSE_ORIGIN  VARCHAR(16777216)  COLUMN     Y    

C:\Users\frase\AppData\Local\Temp\ipykernel_38872\1930253462.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_orders = pd.read_sql(query, conn)


In [12]:
query = """
CREATE OR REPLACE TABLE SALES_STAGING."STG_ORDERS" AS

SELECT
    -- === PRIMARY KEY ===
    order_id,

    -- === FOREIGN KEYS ===
    customer_id,
    seller_id,

    -- === TIMESTAMPS (cast if needed) ===
    TRY_TO_TIMESTAMP(order_date)              AS order_ts,
    TRY_TO_TIMESTAMP(approval_date)           AS approval_ts,
    TRY_TO_TIMESTAMP(shipped_date)            AS shipped_ts,
    TRY_TO_TIMESTAMP(delivered_date)          AS delivered_ts,
    TRY_TO_TIMESTAMP(estimated_delivery_date) AS estimated_delivery_ts,

    -- === DATE PARTS (for grouping) ===
    DATE_TRUNC('month', TRY_TO_TIMESTAMP(order_date))  AS order_month,
    DATE_TRUNC('week',  TRY_TO_TIMESTAMP(order_date))  AS order_week,
    YEAR(TRY_TO_TIMESTAMP(order_date))                  AS order_year,
    QUARTER(TRY_TO_TIMESTAMP(order_date))               AS order_quarter,
    MONTH(TRY_TO_TIMESTAMP(order_date))                 AS order_month_num,
    DAYOFWEEK(TRY_TO_TIMESTAMP(order_date))             AS order_dow,
    HOUR(TRY_TO_TIMESTAMP(order_date))                  AS order_hour,

    -- === LIFECYCLE DURATIONS (core of bottleneck analysis) ===
    DATEDIFF('hour',
        TRY_TO_TIMESTAMP(order_date),
        TRY_TO_TIMESTAMP(approval_date)
    ) AS hours_to_approve,

    DATEDIFF('hour',
        TRY_TO_TIMESTAMP(approval_date),
        TRY_TO_TIMESTAMP(shipped_date)
    ) AS hours_to_ship,

    DATEDIFF('hour',
        TRY_TO_TIMESTAMP(shipped_date),
        TRY_TO_TIMESTAMP(delivered_date)
    ) AS hours_in_transit,

    DATEDIFF('hour',
        TRY_TO_TIMESTAMP(order_date),
        TRY_TO_TIMESTAMP(delivered_date)
    ) AS hours_total_lifecycle,

    -- === DELIVERY PERFORMANCE ===
    CASE
        WHEN TRY_TO_TIMESTAMP(delivered_date) > TRY_TO_TIMESTAMP(estimated_delivery_date)
        THEN 1 ELSE 0
    END AS is_late_delivery,

    DATEDIFF('day',
        TRY_TO_TIMESTAMP(estimated_delivery_date),
        TRY_TO_TIMESTAMP(delivered_date)
    ) AS days_late,

    -- === DATA QUALITY FLAGS ===
    CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END AS flag_missing_customer,

    CASE
        WHEN TRY_TO_TIMESTAMP(delivered_date) < TRY_TO_TIMESTAMP(order_date)
        THEN 1 ELSE 0
    END AS flag_impossible_date,

    CASE
        WHEN order_status = 'Delivered' AND delivered_date IS NULL
        THEN 1 ELSE 0
    END AS flag_missing_delivery_date,

    -- === ORIGINAL FIELDS (pass through as-is) ===
    order_status,
    customer_region,
    customer_city,
    carrier,
    warehouse_origin,
    num_items,
    order_total,
    shipping_cost,
    total_weight_kg,
    payment_method,
    is_weekend_order,
    order_channel

FROM SALES_RAW."01_ORDERS";
"""
cursor = conn.cursor()
cursor.execute(query)
print("✓ Table SALES_STAGING.STG_ORDERS created successfully")
print(f"  Rows affected: {cursor.rowcount}")

✓ Table SALES_STAGING.01_ORDERS created successfully
  Rows affected: 1


---

## 🛒 Staging: `STG_ORDER_ITEMS`

**Source:** `SALES_RAW.02_ORDER_ITEMS` + `SALES_RAW.03_PRODUCTS`

### Transformations

| Category | Description |
|----------|-------------|
| Product Context | Join product details (`category`, `subcategory`, `cost_price`, `is_fragile`, `is_hazardous`) |
| Profitability | Calculate `item_profit` and `margin_pct` per line item |
| Flags | `was_discounted` indicator |

In [13]:
query = """
CREATE OR REPLACE TABLE SALES_STAGING."STG_ORDER_ITEMS" AS

SELECT
    oi.order_item_id,
    oi.order_id,
    oi.product_id,
    oi.seller_id,
    oi.quantity,
    oi.unit_price,
    oi.discount_pct,
    oi.item_total,
    oi.item_weight_kg,

    -- === PRODUCT CONTEXT ===
    p.category,
    p.subcategory,
    p.price          AS list_price,
    p.cost_price,
    p.is_fragile,
    p.is_hazardous,

    -- === PROFITABILITY ===
    ROUND((oi.unit_price - p.cost_price) * oi.quantity, 2)                    AS item_profit,
    ROUND((oi.unit_price - p.cost_price) / NULLIF(oi.unit_price, 0) * 100, 2) AS margin_pct,

    -- === FLAGS ===
    CASE WHEN oi.discount_pct > 0 THEN 1 ELSE 0 END AS was_discounted

FROM SALES_RAW."02_ORDER_ITEMS" oi
LEFT JOIN SALES_RAW."03_PRODUCTS" p
    ON oi.product_id = p.product_id;
"""
cursor = conn.cursor()
cursor.execute(query)
print("✓ Table SALES_STAGING.STG_ORDER_ITEMS created successfully")
print(f"  Rows affected: {cursor.rowcount}")

✓ Table SALES_STAGING.STG_ORDER_ITEMS created successfully
  Rows affected: 1


---

## 🔄 Staging: `STG_RETURNS`

**Source:** `SALES_RAW.07_RETURNS_CANCELLATIONS` + `SALES_RAW.03_PRODUCTS`

### Transformations

| Category | Description |
|----------|-------------|
| Product Context | Join `product_category`, `product_subcategory`, `cost_price` |
| Processing Time | Calculate `return_transit_days` (initiated → received) |
| Costs | Calculate `total_return_cost` (refund + restocking fee) |
| Flags | `flag_not_received` for returns still in transit |

In [14]:
query = """
CREATE OR REPLACE TABLE SALES_STAGING."STG_RETURNS" AS

SELECT
    r.return_id,
    r.order_id,
    r.order_item_id,
    r.product_id,
    r.customer_id,
    r.return_initiated_date,
    r.return_received_date,
    r.return_reason,
    r.return_condition,
    r.refund_amount,
    r.refund_method,
    r.return_status,
    r.restocking_fee,

    -- === PRODUCT CONTEXT ===
    p.category        AS product_category,
    p.subcategory     AS product_subcategory,
    p.cost_price,

    -- === CALCULATED FIELDS ===
    DATEDIFF('day',
        r.return_initiated_date,
        r.return_received_date
    ) AS return_transit_days,

    ROUND(r.refund_amount + COALESCE(r.restocking_fee, 0), 2) AS total_return_cost,

    -- === FLAGS ===
    CASE WHEN r.return_received_date IS NULL THEN 1 ELSE 0 END AS flag_not_received

FROM SALES_RAW."07_RETURNS_CANCELLATIONS" r
LEFT JOIN SALES_RAW."03_PRODUCTS" p
    ON r.product_id = p.product_id;
"""
cursor = conn.cursor()
cursor.execute(query)
print("✓ Table SALES_STAGING.STG_RETURNS created successfully")
print(f"  Rows affected: {cursor.rowcount}")

✓ Table SALES_STAGING.STG_RETURNS created successfully
  Rows affected: 1
